In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point, Polygon
import ipywidgets as widgets
from IPython.display import display, clear_output
import lonboard as lb
from shapely.ops import unary_union, linemerge
import pandas as pd
import networkx as nx
from shapely.geometry import LineString
from shapely.geometry import MultiLineString, Point
from shapely.geometry import MultiPoint

In [2]:
# Load the correct layer
gdf = gpd.read_file("denmark.gpkg", layer="gis_osm_roads_free") 

In [3]:
def road_extractor(gdf):
    # Reproject everything to EPSG:4326 (WGS84) upfront for consistency.
    # lonboard also expects EPSG:4326, so this avoids downstream CRS issues.
    gdf = gdf.to_crs(epsg=4326)

    # Objective 1: Filter to only Secondary, Primary, Tertiary and residential roads
    target_fclasses = ["secondary", "primary", "tertiary", "residential", "unclassified"]
    gdf_filtered = gdf[gdf["fclass"].isin(target_fclasses)].copy()

    gdf_filtered = gdf_filtered[gdf_filtered["name"].notna() & (gdf_filtered["name"] != "")]
    return gdf_filtered.reset_index(drop=True)

gdf_merged = road_extractor(gdf)

In [4]:
def extend_line(line, distance):
    """Extend a LineString by `distance` meters at both ends by extrapolating the end segments."""
    coords = list(line.coords)
    if len(coords) < 2:
        return line

    # Extend start: extrapolate backwards from coords[1] -> coords[0]
    x0, y0 = coords[0]
    x1, y1 = coords[1]
    dx, dy = x0 - x1, y0 - y1
    length = np.hypot(dx, dy)
    if length > 0:
        coords[0] = (x0 + distance * dx / length, y0 + distance * dy / length)

    # Extend end: extrapolate forwards from coords[-2] -> coords[-1]
    x0, y0 = coords[-2]
    x1, y1 = coords[-1]
    dx, dy = x1 - x0, y1 - y0
    length = np.hypot(dx, dy)
    if length > 0:
        coords[-1] = (x1 + distance * dx / length, y1 + distance * dy / length)

    return LineString(coords)


def merge_intersecting_segments(group):
    """
    Within a same-name group, build a graph where each segment is a node.
    Add an edge between two segments if they intersect or touch.
    Merge only the segments within each connected component.
    Returns a list of (row, merged_geometry) tuples — one per connected component.
    """
    geoms = list(group.geometry)
    n = len(geoms)

    G = nx.Graph()
    G.add_nodes_from(range(n))

    for i in range(n):
        for j in range(i + 1, n):
            if geoms[i].intersects(geoms[j]) or geoms[i].touches(geoms[j]):
                G.add_edge(i, j)

    rows = []
    for component in nx.connected_components(G):
        indices = list(component)
        merged_geom = unary_union([geoms[i] for i in indices])
        row = group.iloc[indices[0]].copy()
        row["geometry"] = merged_geom
        component_fclasses = group.iloc[indices]["fclass"].unique()
        row["fclass"] = ", ".join(sorted(component_fclasses))
        rows.append(row)

    return rows


def filtered_frame(gdf):
    # Drop rows with no name
    gdf = gdf[gdf["name"].notna() & (gdf["name"] != "")].copy()

    # Reproject to metric CRS (EPSG:3857) so 1 meter extension is accurate
    gdf_metric = gdf.to_crs(epsg=3857)

    # Explode MultiLineStrings into individual LineStrings
    gdf_exploded = gdf_metric.explode(index_parts=False).reset_index(drop=True)
    gdf_exploded = gdf_exploded[gdf_exploded.geometry.type == "LineString"].copy()

    # Extend each segment by x meter at both ends so near-touching segments overlap
    gdf_exploded["geometry"] = gdf_exploded.geometry.apply(lambda line: extend_line(line, 0.0))

    # For each name group, merge only connected (intersecting/touching) segments
    all_rows = []
    for name, group in gdf_exploded.groupby("name"):
        all_rows.extend(merge_intersecting_segments(group))

    gdf_merged = gpd.GeoDataFrame(all_rows, geometry="geometry", crs="EPSG:3857").reset_index(drop=True)

    # Reproject back to WGS84
    return gdf_merged.to_crs(epsg=4326)


In [5]:
roadnetwork_filtered = filtered_frame(gdf_merged)

#roadnetwork_filtered.to_file(r'd:\\Pipe_builder_gui\\road_network_postproc.geojson', driver='GeoJSON')

In [6]:
from shapely.geometry import Point

def find_road_intersections(gdf):
    """
    Find intersections between different roads in a GeoDataFrame.
    Uses a spatial index (STRtree) to avoid O(n²) pair comparisons.
    """
    gdf_metric = gdf.to_crs(epsg=3857).reset_index(drop=True)

    # Spatial index: only candidate pairs whose bounding boxes overlap are checked
    sindex = gdf_metric.sindex
    left_idx, right_idx = sindex.query(gdf_metric.geometry, predicate="intersects")

    # Keep unique pairs (i < j) — drops self-pairs and duplicates
    mask = left_idx < right_idx
    left_idx = left_idx[mask]
    right_idx = right_idx[mask]

    # Pre-filter pairs with the same road name
    names = gdf_metric.get("name", pd.Series([""] * len(gdf_metric))).fillna("").values
    name_mask = names[left_idx] != names[right_idx]
    left_idx = left_idx[name_mask]
    right_idx = right_idx[name_mask]

    geometries = gdf_metric.geometry.values
    intersection_points = []

    for i, j in zip(left_idx, right_idx):
        geom_a = geometries[i]
        geom_b = geometries[j]

        intersection = geom_a.intersection(geom_b)
        if intersection.is_empty:
            continue

        pts = []
        if intersection.geom_type == "Point":
            pts = [intersection]
        elif intersection.geom_type == "MultiPoint":
            pts = list(intersection.geoms)
        elif intersection.geom_type in ("GeometryCollection", "MultiLineString", "LineString"):
            if hasattr(intersection, "geoms"):
                for g in intersection.geoms:
                    pts.append(g if g.geom_type == "Point" else g.centroid)
            else:
                pts.append(intersection.centroid)

        if not pts:
            continue

        road_a = gdf_metric.iloc[i]
        road_b = gdf_metric.iloc[j]
        name_a = road_a.get("name", road_a.get("osm_id", str(i)))
        name_b = road_b.get("name", road_b.get("osm_id", str(j)))
        fclass_a = road_a.get("fclass", "")
        fclass_b = road_b.get("fclass", "")
        osm_a = road_a.get("osm_id", "")
        osm_b = road_b.get("osm_id", "")

        for pt in pts:
            intersection_points.append({
                "geometry": pt,
                "road_a_name": name_a,
                "road_a_osm_id": osm_a,
                "road_a_fclass": fclass_a,
                "road_b_name": name_b,
                "road_b_osm_id": osm_b,
                "road_b_fclass": fclass_b,
                "intersecting_roads": f"{name_a} ({fclass_a}, osm_id={osm_a}) x {name_b} ({fclass_b}, osm_id={osm_b})"
            })

    if not intersection_points:
        print("No intersections found.")
        return gpd.GeoDataFrame(
            columns=["geometry", "road_a_name", "road_a_osm_id", "road_a_fclass",
                     "road_b_name", "road_b_osm_id", "road_b_fclass", "intersecting_roads"],
            geometry="geometry", crs="EPSG:3857"
        )

    intersections_gdf = gpd.GeoDataFrame(intersection_points, geometry="geometry", crs="EPSG:3857")
    return intersections_gdf.to_crs(epsg=4326)


intersections = find_road_intersections(roadnetwork_filtered)
print(f"Found {len(intersections)} intersection(s)")
#intersections.to_file("road_intersections.geojson", driver="GeoJSON")
#lb.viz(intersections)


Found 169624 intersection(s)


In [7]:
from shapely.ops import unary_union

def cluster_nearby_intersections(intersections_gdf, radius_m=10):
    """
    Cluster intersection points within `radius_m` meters of each other.
    Merges nearby points into a single centroid, combining road info from all.
    """
    gdf = intersections_gdf.to_crs(epsg=3857).copy().reset_index(drop=True)
    
    # Buffer each point by radius_m/2 so overlapping buffers indicate proximity
    buffered = gdf.geometry.buffer(radius_m / 2)
    
    # Build union clusters using spatial index
    sindex = buffered.sindex
    visited = set()
    clusters = []

    for i in range(len(gdf)):
        if i in visited:
            continue
        # Find all points whose buffers intersect with point i's buffer
        candidates = list(sindex.query(buffered.iloc[i], predicate="intersects"))
        cluster = [c for c in candidates if c not in visited]
        for c in cluster:
            visited.add(c)
        clusters.append(cluster)

    merged_rows = []
    for cluster in clusters:
        rows = gdf.iloc[cluster]
        # Centroid of all points in the cluster
        centroid = unary_union(rows.geometry).centroid

        # Collect all unique roads involved
        roads_a = set(zip(rows["road_a_name"], rows["road_a_osm_id"], rows["road_a_fclass"]))
        roads_b = set(zip(rows["road_b_name"], rows["road_b_osm_id"], rows["road_b_fclass"]))
        all_roads = roads_a | roads_b

        road_strs = [f"{name} ({fclass}, osm_id={osm_id})" for name, osm_id, fclass in sorted(all_roads)]
        intersecting_roads_str = " x ".join(road_strs)

        road_list = sorted(all_roads)
        merged_rows.append({
            "geometry": centroid,
            "road_a_name": road_list[0][0] if len(road_list) > 0 else "",
            "road_a_osm_id": road_list[0][1] if len(road_list) > 0 else "",
            "road_a_fclass": road_list[0][2] if len(road_list) > 0 else "",
            "road_b_name": road_list[1][0] if len(road_list) > 1 else "",
            "road_b_osm_id": road_list[1][1] if len(road_list) > 1 else "",
            "road_b_fclass": road_list[1][2] if len(road_list) > 1 else "",
            "intersecting_roads": intersecting_roads_str,
            "num_roads": len(all_roads),
            "cluster_size": len(cluster),
        })

    result = gpd.GeoDataFrame(merged_rows, geometry="geometry", crs="EPSG:3857")
    result = result.to_crs(epsg=4326)
    return result


intersections_clustered = cluster_nearby_intersections(intersections, radius_m=5)
print(f"Before clustering: {len(intersections)} intersections")
print(f"After clustering:  {len(intersections_clustered)} intersections")


Before clustering: 169624 intersections
After clustering:  147345 intersections


In [8]:
intersections_clustered.head()

,geometry,road_a_name,road_a_osm_id,road_a_fclass,road_b_name,road_b_osm_id,road_b_fclass,intersecting_roads,num_roads,cluster_size
0,POINT (9.48713 55.357),10. Februar Vej,89715790,residential,Christian VII's Vej,244059143,residential,"10. Februar Vej (residential, osm_id=89715790)...",2,1
1,POINT (9.48673 55.35892),10. Februar Vej,89715790,residential,Borgmestervej,89715779,residential,"10. Februar Vej (residential, osm_id=89715790)...",2,1
2,POINT (9.4884 55.35709),10. Juli Vej,89715787,residential,Christian VII's Vej,244059143,residential,"10. Juli Vej (residential, osm_id=89715787) x ...",2,1
3,POINT (9.48797 55.35901),10. Juli Vej,89715787,residential,Borgmestervej,89715779,residential,"10. Juli Vej (residential, osm_id=89715787) x ...",2,1
4,POINT (11.38222 54.6833),2. Tværvej,24231217,unclassified,Havnevej,537204554,"secondary, tertiary","2. Tværvej (unclassified, osm_id=24231217) x H...",2,1


In [9]:
if "intersection_id" not in intersections_clustered.columns:
    intersections_clustered["intersection_id"] = intersections_clustered.index

print(intersections_clustered["intersection_id"])

0              0
1              1
2              2
3              3
4              4
           ...  
147340    147340
147341    147341
147342    147342
147343    147343
147344    147344
Name: intersection_id, Length: 147345, dtype: int64


In [10]:
import json

# Load the existing road network postproc geojson
#postproc_path = r'd:\\Pipe_builder_gui\\road_network_postproc.geojson'
#with open(postproc_path, 'r', encoding='utf-8') as f:
#    postproc_geojson = json.load(f)

# Convert filtered_frame output to GeoJSON features
filtered_frame_geojson = json.loads(roadnetwork_filtered.to_json())

# Convert intersections_clustered to GeoJSON features
intersections_geojson = json.loads(intersections_clustered.to_json())

# Tag intersection features with a type indicator and stable id
for feature in intersections_geojson['features']:
    feature['properties']['feature_type'] = 'intersection'
    feature['properties']['id'] = feature['properties'].get('intersection_id')
    feature['id'] = feature['properties']['id']

# Tag road features with a type indicator; keep id as NaN in GeoDataFrame
for feature in filtered_frame_geojson['features']:
    feature['properties']['feature_type'] = 'road'
    feature['properties']['id'] = None
    feature['id'] = None

# Combine features
combined_geojson = {
    "type": "FeatureCollection",
    "features": filtered_frame_geojson['features'] + intersections_geojson['features']
}


#output_path = r'road_network_postproc_PSTR.geojson'
#with open(output_path, 'w', encoding='utf-8') as f:
#    json.dump(combined_geojson, f, ensure_ascii=False)

print(f"Saved {len(filtered_frame_geojson['features'])} roads + {len(intersections_geojson['features'])} intersections to combined GeoJSON")

Saved 109042 roads + 147345 intersections to combined GeoJSON


In [11]:
# make combined_geojson from dict to geojson dataframe
combined_geojson = gpd.GeoDataFrame.from_features(combined_geojson['features'])

In [12]:
combined_geojson.head()

# find the id column in combined_geojson
#print("Columns in combined_geojson:", combined_geojson.columns)

,geometry,osm_id,code,fclass,name,ref,oneway,maxspeed,layer,bridge,...,road_a_name,road_a_osm_id,road_a_fclass,road_b_name,road_b_osm_id,road_b_fclass,intersecting_roads,num_roads,cluster_size,intersection_id
0,"LINESTRING (9.48673 55.35892, 9.48713 55.357)",89715790,5122.0,residential,10. Februar Vej,,B,0.0,0.0,F,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"LINESTRING (9.48797 55.35901, 9.4884 55.35709)",89715787,5122.0,residential,10. Juli Vej,,B,0.0,0.0,F,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"LINESTRING (11.36817 54.68503, 11.37967 54.683...",24231217,5121.0,unclassified,2. Tværvej,,B,80.0,0.0,F,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"LINESTRING (11.36658 54.67974, 11.3722 54.6789...",24231220,5121.0,unclassified,3. Tværvej,,B,80.0,0.0,F,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"MULTILINESTRING ((10.51571 54.8535, 10.51599 5...",27922650,5122.0,residential,4. Maj Stræde,,B,0.0,0.0,F,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# read the road network data
combined_geojson = gpd.read_file(r'd:\\Pipe_builder_gui\\road_network_postproc_PSTR.geojson')

In [14]:

import math
# =========================================================================== #
#  Constants                                                                   #
# =========================================================================== #

LINE_TYPES  = (LineString, MultiLineString)
POINT_TYPES = (Point, MultiPoint)


# =========================================================================== #
#  Internal helpers                                                            #
# =========================================================================== #

def _is_null(value) -> bool:
    """Return True for None, empty string, or float NaN."""
    if value is None or value == "":
        return True
    try:
        return math.isnan(float(value))
    except (TypeError, ValueError):
        return False


def _build_projector(gdf: gpd.GeoDataFrame,
                     svg_w: float = 1200,
                     svg_h: float = 900,
                     padding: float = 40.0):
    """
    Return a (lon, lat) -> (svg_x, svg_y) projection function derived from
    the bounding box of the entire GeoDataFrame.

    - Y-axis is flipped  (SVG origin = top-left)
    - Aspect ratio is preserved via uniform scaling
    - The map is centred inside the viewport with the requested padding
    
    Returns:
        tuple: (project_function, metadata_dict) where metadata includes scale and bounds
    """
    min_x, min_y, max_x, max_y = gdf.geometry.total_bounds
    print(f"GeoDataFrame bounds: min_x={min_x}, min_y={min_y}, max_x={max_x}, max_y={max_y}")

    usable_w = svg_w - 2 * padding
    usable_h = svg_h - 2 * padding
    span_x   = max_x - min_x or 1e-9
    span_y   = max_y - min_y or 1e-9
    scale    = min(usable_w / span_x, usable_h / span_y)
    print(f'scale = {scale}')

    offset_x = padding + (usable_w - span_x * scale) / 2
    offset_y = padding + (usable_h - span_y * scale) / 2

    def project(lon: float, lat: float) -> tuple:
        x = round(offset_x + (lon - min_x) * scale, 3)
        y = round(offset_y + (max_y - lat) * scale, 3)   # flip Y
        
        return x, y

    metadata = {
        "scale": scale,
        "bounds": {
            "min_x": min_x,
            "min_y": min_y,
            "max_x": max_x,
            "max_y": max_y
        },
        "svg_dimensions": {
            "width": svg_w,
            "height": svg_h,
            "used_width": usable_w,
            "used_height": usable_h,
            "padding": padding
        },
        "offset": {
            "x": offset_x,
            "y": offset_y
        }
    }

    return project, metadata


def _linestring_to_svg_path(coords, project) -> str:
    """
    Convert a coordinate sequence to an SVG path 'd' string.
    e.g. "M10,20 L30,40 L50,60"
    """
    parts = []
    for i, (lon, lat) in enumerate(coords):
        x, y = project(lon, lat)
        parts.append(f"{'M' if i == 0 else 'L'}{x},{y}")
    return " ".join(parts)


def _geom_to_path_details(geom, project) -> list:
    """
    Return a list of SVG path 'd' strings for a LineString or MultiLineString.
    Each sub-linestring in a MultiLineString becomes its own entry.
    """
    if isinstance(geom, LineString):
        return [_linestring_to_svg_path(geom.coords, project)]
    if isinstance(geom, MultiLineString):
        return [_linestring_to_svg_path(line.coords, project)
                for line in geom.geoms]
    return []


def _geom_to_point(geom, project) -> list:
    """
    Return a list of {"cx": x, "cy": y} dicts for a Point or MultiPoint.
    """
    if isinstance(geom, Point):
        cx, cy = project(geom.x, geom.y)
        return [{"cx": cx, "cy": cy}]
    if isinstance(geom, MultiPoint):
        return [{"cx": project(pt.x, pt.y)[0], "cy": project(pt.x, pt.y)[1]}
                for pt in geom.geoms]
    return []


# =========================================================================== #
#  Main export function                                                        #
# =========================================================================== #

def gdf_to_road_intersection_json(
        gdf: gpd.GeoDataFrame,
        output_path: str   = "output.json",
        svg_width:   int   = 1200,
        svg_height:  int   = 900,
        padding:     float = 40.0) -> dict:
    """
    Build a roads <-> intersections mapping with embedded SVG geometry
    and write it as a single JSON file.

    Geometry-aware column mapping
    ─────────────────────────────
    LineString / MultiLineString  ->  Road_ID entry
        'osm_id'        used as the key
        'path_details'  list of SVG path 'd' strings (one per sub-linestring)

    Point / MultiPoint            ->  intersection_ID entry
        'id'            used as the key
        'road_a_osm_id' / 'road_b_osm_id'  -> road_IDs list
        'point'         list of {"cx": x, "cy": y} SVG coordinates

    Output schema
    ─────────────
    {
      "metadata": {...},
      "Road_ID": {
        "<osm_id>": {
          "intersection_IDs": ["<id>", ...],
          "path_details":     ["M x,y L x,y ...", ...]
        }
      },
      "intersection_ID": {
        "<id>": {
          "road_IDs": ["<osm_id_a>", "<osm_id_b>"],
          "point":    [{"cx": x, "cy": y}]
        }
      }
    }

    Parameters
    ──────────
    gdf         : Input GeoDataFrame (geographic CRS recommended).
    output_path : Destination JSON file path.
    svg_width   : Viewport width  used to compute SVG coordinates (default 1200).
    svg_height  : Viewport height used to compute SVG coordinates (default 900).
    padding     : Margin (px) around the map inside the viewport  (default 40).
    """

    project, metadata = _build_projector(gdf, svg_width, svg_height, padding)

    road_section:         dict = {}
    intersection_section: dict = {}

    for _, row in gdf.iterrows():
        geom = row.get("geometry")
        if geom is None:
            continue

        # ── Roads (LineString / MultiLineString) ───────────────────────────── #
        if isinstance(geom, LINE_TYPES):
            osm_id = str(row["osm_id"])
            if osm_id not in road_section:
                road_section[osm_id] = {
                    "intersection_IDs": [],
                    "path":     _geom_to_path_details(geom, project),
                }

        # ── Intersections (Point / MultiPoint) ────────────────────────────── #
        elif isinstance(geom, POINT_TYPES):
            int_id = str(row["id"])

            road_osm_ids = (
                ([str(row["road_a_osm_id"])] if not _is_null(row.get("road_a_osm_id")) else []) +
                ([str(row["road_b_osm_id"])] if not _is_null(row.get("road_b_osm_id")) else [])
            )

            intersection_section[int_id] = {
                "road_IDs": road_osm_ids,
                "point":    _geom_to_point(geom, project),
            }

            # Back-fill each road's intersection_IDs list
            for osm_id in road_osm_ids:
                if osm_id not in road_section:
                    road_section[osm_id] = {"intersection_IDs": [], "path": []}
                if int_id not in road_section[osm_id]["intersection_IDs"]:
                    road_section[osm_id]["intersection_IDs"].append(int_id)

    result = {
        "metadata": metadata,
        "Road_ID":         road_section,
        "intersection_ID": intersection_section,
    }

    with open(output_path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"JSON written to: {output_path}")
    return result

In [15]:
# export the combined_geojson to the desired JSON format

result = gdf_to_road_intersection_json(combined_geojson, "roadnetwork_postproc_PSTR.json")

GeoDataFrame bounds: min_x=8.084258019354836, min_y=54.561135672924806, max_x=15.1529647, max_y=57.75046267473295
scale = 158.44482599153162
JSON written to: roadnetwork_postproc_PSTR.json


In [16]:
import re
def result_to_js(result: dict,
                output_path: str = "processedRoads.js",
                variable_name: str = "RAW") -> dict:
    """
    Convert the generated result dict into processedRoads.js format for the HTML GUI.
    Exports: roads (SVG paths), intersections (coordinates), roads_bb (bounding boxes), and metadata.
    
    Output format:
    const RAW = {
        "metadata": {...scale, bounds, etc...},
        "roads": {"<osm_id>": "M x,y L x,y ..."},
        "intersections": {"<int_id>": [cx, cy]},
        "roads_bb": {"<osm_id>": [x1, y1, x2, y2]}
    };
    """
    metadata_in = result.get("metadata", {}) if isinstance(result, dict) else {}
    roads_in = result.get("Road_ID", {}) if isinstance(result, dict) else {}
    intersections_in = result.get("intersection_ID", {}) if isinstance(result, dict) else {}
    
    roads_out = {}
    intersections_out = {}
    roads_bb_out = {}

    # Process roads and calculate bounding boxes
    for osm_id, road_info in roads_in.items():
        if not isinstance(road_info, dict):
            continue

        # Each road may contain multiple path fragments; join them into one string
        path_parts = road_info.get("path", [])
        if isinstance(path_parts, list):
            joined_path = " ".join(p for p in path_parts if isinstance(p, str) and p.strip())
        elif isinstance(path_parts, str):
            joined_path = path_parts
        else:
            joined_path = ""

        if joined_path:
            roads_out[str(osm_id)] = joined_path
            
            # Calculate bounding box from SVG path commands (M x,y L x,y ...)
            coords = []
            for match in re.finditer(r'([ML])\s*([\d.]+)[,\s]+([\d.]+)', joined_path):
                coords.append((float(match.group(2)), float(match.group(3))))
            
            if coords:
                xs = [c[0] for c in coords]
                ys = [c[1] for c in coords]
                roads_bb_out[str(osm_id)] = [min(xs), min(ys), max(xs), max(ys)]

    # Process intersections
    for int_id, int_info in intersections_in.items():
        if not isinstance(int_info, dict):
            continue
        
        point_data = int_info.get("point", [])
        if isinstance(point_data, list) and len(point_data) > 0:
            # Extract first point [cx, cy]
            first_point = point_data[0]
            if isinstance(first_point, dict):
                cx = first_point.get("cx")
                cy = first_point.get("cy")
                if cx is not None and cy is not None:
                    intersections_out[str(int_id)] = [cx, cy]

    payload = {
        "metadata": metadata_in,
        "roads": roads_out,
        "intersections": intersections_out,
        "roads_bb": roads_bb_out
    }

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(f"const {variable_name} = ")
        json.dump(payload, f, separators=(",", ":"), ensure_ascii=False)
        f.write(";\n")
    return payload

In [17]:
# result to js
js_payload = result_to_js(result, "processedRoads.js", "RAW")